# Building the simplest possible Agent with the latest LangChain

This document is a step-by-step guide that uses the `create_agent` API of the
**latest LangChain (the 1.x line)**, covering everything from installation to a working example.

> Key point: the old `initialize_agent`, `AgentExecutor`, and `create_react_agent`
> approaches are no longer recommended. The current standard is to build a
> tool-using Agent with a **single** call to `langchain.agents.create_agent`.

## 1. Prerequisites

- **Python 3.10 or higher**
- An API key for an LLM provider (the examples below assume OpenAI or Anthropic)

In [4]:
!python --version

Python 3.14.5


## 2. Installation



### Installing LangChain

Install the extra that matches the model provider you will use.

> Writing it in brackets like `langchain[openai]` automatically installs the
> required integration packages (such as `langchain-openai`) alongside it.

In [11]:
%pip install -U langchain "langchain[openai]"

Note: you may need to restart the kernel to use updated packages.


## 3. Setting the API key

Do not hardcode the key in your code; set it as an environment variable.

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."

In [ ]:
print(os.environ.get("OPENAI_API_KEY"))

## 4. The simplest Agent example

In [17]:
from langchain.agents import create_agent

# 1) Define the tool(s) the Agent will use
#    - The function's docstring becomes the tool description, so always write one.
#    - The LLM fills in the arguments based on the type hints.
def get_weather(city: str) -> str:
    """Tells you the weather for the given city."""
    return f"The weather in {city} is always sunny!"

# 2) Create the Agent
agent = create_agent(
    model="openai:gpt-5.4",          # format: "<provider>:<model>"
    tools=[get_weather],
    system_prompt="You are a friendly assistant.",
)

# 3) Run the Agent
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)

# 4) Print the final response
print(result["messages"][-1].content)

San Francisco is always sunny! ☀️


## 5. Running it
Expected output (summary):

```
The weather in San Francisco is always sunny!
```

Internally, the Agent automatically performs the following steps:

1. Receives the user's question
2. Decides that it should call the `get_weather` tool
3. Runs the tool with `city="San Francisco"`
4. Takes the tool result and generates a natural-language answer

## 6. One step further: multiple tools + the `@tool` decorator

When you have many tools, declaring them explicitly with the `@tool` decorator is cleaner.

In [18]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Tells you the weather for the given city."""
    return f"The weather in {city} is sunny!"

@tool
def add(a: int, b: int) -> int:
    """Adds two integers."""
    return a + b


agent = create_agent(
    model="openai:gpt-5.4",
    tools=[get_weather, add],
    system_prompt="You are a friendly assistant. Use tools when needed.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Tell me the weather in Seoul, and also compute 3 plus 5."}]}
)
print(result["messages"][-1].content)

The weather in Seoul is sunny!  
3 plus 5 is 8.


## 7. Streaming output

If you want to receive the response in real time, use `stream`.

In [19]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What's the weather in Tokyo?"}]},
    stream_mode="values",
):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

What's the weather in Tokyo?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_MhuxTBCs8wBk6q6jz6UZWa1y)
 Call ID: call_MhuxTBCs8wBk6q6jz6UZWa1y
  Args:
    city: Tokyo
================================= Tool Message =================================
Name: get_weather

The weather in Tokyo is sunny!
================================== Ai Message ==================================

Tokyo is currently sunny!
